In [30]:
import pandas as pd

In [31]:
train_df = pd.read_csv('../dados/train.csv')
test_df = pd.read_csv('../dados/test.csv')

In [32]:
print("Shape do treino processado:", train_df.shape)
print("Shape do teste processado:", test_df.shape)

Shape do treino processado: (891, 12)
Shape do teste processado: (418, 11)


In [33]:
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [34]:
train_df.info()
train_df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [35]:
from pandasql import sqldf
pysqldf = lambda q: sqldf(q, globals())

query = """
    SELECT Pclass, Sex, Embarked, count(distinct PassengerId) as total
    FROM train_df
    WHERE Age is null
    group by 1,2,3
    order by total desc
"""

resultado = pysqldf(query)
display(resultado)

,Pclass,Sex,Embarked,total
0,3,male,S,51
1,3,male,Q,25
2,3,female,Q,23
3,3,male,C,18
4,1,male,S,15
5,3,female,S,12
6,2,male,S,7
7,3,female,C,7
8,1,male,C,6
9,1,female,C,5


In [36]:
pysqldf = lambda q: sqldf(q, globals())
query = """
    SELECT PClass, Sex, Embarked, max(age) as idade_max, min(age) as idade_min, median(age) as idade_mediana
    FROM train_df
    WHERE age is not null
    group by 1,2,3
"""

resultado = pysqldf(query)
display(resultado)

,Pclass,Sex,Embarked,idade_max,idade_min,idade_mediana
0,1,female,NaN,62.0,38.00,50.00
1,1,female,C,60.0,16.00,37.00
2,1,female,Q,33.0,33.00,33.00
3,1,female,S,63.0,2.00,33.00
4,1,male,C,71.0,17.00,36.50
5,1,male,Q,44.0,44.00,44.00
6,1,male,S,80.0,0.92,42.00
7,2,female,C,28.0,3.00,22.00
8,2,female,Q,30.0,30.00,30.00
9,2,female,S,57.0,2.00,29.00


In [37]:
pysqldf = lambda q: sqldf(q, globals())
query = """
    SELECT PClass, Embarked, SUBSTR(Cabin, 1, 1) AS Deck, count(distinct PassengerId) as total
    FROM train_df
    where Cabin is not null
    group by 1,2,3
    order by 4 desc
"""

resultado = pysqldf(query)
display(resultado)

,Pclass,Embarked,Deck,total
0,1,S,C,36
1,1,S,B,23
2,1,C,B,22
3,1,C,C,21
4,1,S,E,20
5,1,S,D,18
6,1,C,D,11
7,1,S,A,8
8,2,S,F,8
9,1,C,A,7


In [38]:
def renomear_colunas(df):
    renomes = {
        'PassengerId': 'Id_Passageiro',
        'Survived': 'Sobreviveu',
        'Pclass': 'ClassePassageiro',
        'Name': 'Nome',
        'Sex': 'Sexo',
        'Age': 'Idade',
        'SibSp': 'Irmaos_Conjuges',
        'Parch': 'Pais_Filhos',
        'Ticket': 'NumeroBilhete',
        'Fare': 'ValorBilhete',
        'Cabin': 'Cabine',
        'Embarked': 'Porto_Embarque',
    }
    return df.rename(columns=renomes)

def preencher_idade(df, medianas_referencia):
    def aplicar_mediana(linha):
        if pd.isnull(linha['Idade']):
            try:
                return medianas_referencia[linha['ClassePassageiro'], linha['Sexo'], linha['Porto_Embarque']]
            except KeyError:
                return medianas_referencia.median() # Plano B de segurança
        return linha['Idade']

    df = df.copy()
    df['Idade'] = df.apply(aplicar_mediana, axis=1)
    return df

def identificar_deck(df):
    df = df.copy()
    df['Deck'] = df['Cabine'].str[0].fillna('U')
    df = df.drop('Cabine', axis=1)
    return df

def preencher_porto(df, valor_preenchimento):
    df = df.copy()
    df['Porto_Embarque'] = df['Porto_Embarque'].fillna(valor_preenchimento)
    return df

def preencher_tarifa(df, mediana_referencia):
    def aplicar_fare(linha):
        if pd.isnull(linha['ValorBilhete']):
            return mediana_referencia[linha['ClassePassageiro']]
        return linha['ValorBilhete']

    df = df.copy()
    df['ValorBilhete'] = df.apply(aplicar_fare, axis=1)
    return df

def transformacoes_v2(df):
    df = df.copy()
    
    # Mapeamento binário do sexo
    df['Sexo'] = df['Sexo'].map({'male': 0, 'female': 1})

    # criação de colunas para familia
    df['Total_Familiar'] = df['Irmaos_Conjuges'] + df['Pais_Filhos'] + 1
    df['Sozinho'] = (df['Total_Familiar'] == 1).astype(int)

    # One-Hot Encoding para variáveis categóricas
    df = pd.get_dummies(df, columns=['Porto_Embarque', 'Deck'], drop_first=True)
    
    # Descarte de colunas que não serão usadas
    df = df.drop(columns=['Nome', 'NumeroBilhete'], errors='ignore')

    # Ajuste de tipos
    colunas_booleanas = df.select_dtypes(include=['bool']).columns
    df[colunas_booleanas] = df[colunas_booleanas].astype(int)
    
    return df

In [39]:
# renomear colunas de ambos os datasets
train_df = renomear_colunas(train_df)
test_df = renomear_colunas(test_df)

# calcular parâmetros do treino
medianas_idade_treino = train_df.groupby(['ClassePassageiro', 'Sexo', 'Porto_Embarque'])['Idade'].median()
porto_moda_treino = train_df['Porto_Embarque'].mode()[0]
mediana_fare_treino = train_df.groupby('ClassePassageiro')['ValorBilhete'].median()

# aplicar limpezas no treino
train_df = preencher_idade(train_df, medianas_idade_treino)
train_df = identificar_deck(train_df)
train_df = preencher_porto(train_df, porto_moda_treino)
train_df = preencher_tarifa(train_df, mediana_fare_treino)

# aplicar regras no teste
test_df = preencher_idade(test_df, medianas_idade_treino)
test_df = identificar_deck(test_df)
test_df = preencher_porto(test_df, porto_moda_treino)
test_df = preencher_tarifa(test_df, mediana_fare_treino)

In [40]:
def transformacoes_v2(df):
    df['Sexo'] = df['Sexo'].map({'male': 0, 'female': 1})

    df['Total_Familiar'] = df['Irmaos_Conjuges'] + df['Pais_Filhos'] + 1

    df['Sozinho'] = (df['Total_Familiar'] == 1).astype(int)
    df = pd.get_dummies(df, columns=['Porto_Embarque', 'Deck'], drop_first=True)
    df = df.drop(columns=['NumeroBilhete'])

    #ajustar tipos
    colunas_booleanas = df.select_dtypes(include=['bool']).columns
    df[colunas_booleanas] = df[colunas_booleanas].astype(int)
    
    return df

In [41]:
train_df = transformacoes_v2(train_df)
test_df = transformacoes_v2(test_df)

In [42]:
train_df.head()

,Id_Passageiro,Sobreviveu,ClassePassageiro,Nome,Sexo,Idade,Irmaos_Conjuges,Pais_Filhos,ValorBilhete,Total_Familiar,...,Porto_Embarque_Q,Porto_Embarque_S,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,7.2500,2,...,0,1,0,0,0,0,0,0,0,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,71.2833,2,...,0,0,0,1,0,0,0,0,0,0
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,7.9250,1,...,0,1,0,0,0,0,0,0,0,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,53.1000,2,...,0,1,0,1,0,0,0,0,0,0
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,8.0500,1,...,0,1,0,0,0,0,0,0,0,1


In [43]:
train_df.to_csv('../dados/train_processado.csv', index=False)
test_df.to_csv('../dados/test_processado.csv', index=False)